# Ingeniería de Prompts: Técnicas y Aplicaciones

Este cuaderno explora técnicas avanzadas de ingeniería de prompts para modelos de lenguaje, con ejemplos prácticos y análisis crítico en cada sección.

## 1. Sección de Configuración

En esta sección se configuran las variables de entorno necesarias y se define la función auxiliar `get_completion` para interactuar con el modelo de lenguaje.

In [47]:
# Environment setup and robust get_completion helper function
import os
from dotenv import load_dotenv
import openai
from typing import Optional, Dict, Any

# Load environment variables from .env file
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE")
OPENAI_API_TYPE = os.getenv("OPENAI_API_TYPE", "azure")
OPENAI_API_VERSION = os.getenv("OPENAI_API_VERSION", "2023-05-15")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME")

if OPENAI_API_TYPE == "azure":
    client = openai.AzureOpenAI(
        api_key=OPENAI_API_KEY,
        azure_endpoint=f"{OPENAI_API_BASE}",
        api_version=OPENAI_API_VERSION,
    )
else:
    client = openai.OpenAI(api_key=OPENAI_API_KEY)

def get_completion(prompt: str, temperature: float = 0, max_tokens: int = 512, \
                  deployment_name: Optional[str] = None, \
                  system_prompt: Optional[str] = None, \
                  extra_params: Optional[Dict[str, Any]] = None) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=deployment_name or DEPLOYMENT_NAME,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        **(extra_params or {})
    )
    return response.choices[0].message.content.strip()

## 1.1 Sección: Técnicas de Ingeniería de Prompts

A continuación se presentan seis técnicas fundamentales de ingeniería de prompts, cada una explicada y ejemplificada.

### Técnica 1: Rol / Persona

Permite asignar un rol o personalidad específica al modelo para obtener respuestas alineadas con ese contexto. Esto mejora la relevancia y el tono de las respuestas generadas.

In [48]:
# Example: Role / Persona (Senior Restaurant Consultant)
role_system_prompt = "You are a Senior Restaurant Consultant. Provide professional advice to improve restaurant operations."
user_prompt = "Our restaurant has slow table turnover and frequent customer complaints about waiting times. What should we do?"

response = get_completion(user_prompt, system_prompt=role_system_prompt)
print(response)

Improving table turnover and addressing customer complaints about waiting times requires a multi-faceted approach that focuses on operational efficiency, staff training, and customer experience. Here’s a detailed action plan:

---

### **1. Optimize Front-of-House Operations**
- **Streamline Seating Procedures:**
  - Use a reservation system or waitlist app (e.g., OpenTable, Resy, or Yelp Waitlist) to manage table assignments efficiently.
  - Train hosts to seat guests promptly and balance table assignments across servers to avoid bottlenecks.

- **Implement Clear Table Management Policies:**
  - Set expectations for table turnover times based on meal type (e.g., 60–90 minutes for dinner, 30–45 minutes for lunch).
  - Use subtle cues to signal the end of a meal, such as clearing plates promptly or presenting the check when appropriate.

- **Pre-Set Tables:**
  - Pre-set tables with basic items (e.g., menus, water glasses, or napkins) to reduce setup time between seatings.

---

### **2

**Análisis Crítico:**

**¿Por qué funciona?**
La técnica de rol o persona orienta al modelo para que responda desde una perspectiva experta y contextualizada. Al definir un rol claro, el modelo adapta su lenguaje, profundidad y recomendaciones, logrando respuestas más útiles y alineadas con el objetivo.

**Ventaja competitiva:**
Supera a un prompt genérico porque reduce ambigüedades y sesgos, logrando respuestas más profesionales y relevantes para el dominio.

**Mi contexto:**
Utilizaría esta técnica para simular consultores expertos en restaurantes, obteniendo recomendaciones precisas para mejorar la experiencia del cliente o la eficiencia operativa.

### Técnica 2: Few-Shot

Consiste en proporcionar ejemplos explícitos en el prompt para guiar al modelo en tareas de clasificación, generación o transformación de texto. Esto mejora la precisión en tareas específicas.

In [49]:
# Example: Few-Shot Classification (Restaurant Service Feedback)
few_shot_prompt = '''Classify the following restaurant feedback as Positive, Neutral, or Negative.\n\nExample 1:\nFeedback: "The waiter was very attentive and friendly."\nSentiment: Positive\n\nExample 2:\nFeedback: "The food was okay, but nothing special."\nSentiment: Neutral\n\nExample 3:\nFeedback: "We waited 40 minutes for our order and the staff ignored us."\nSentiment: Negative\n\nFeedback: "The staff was polite but the service was slow."\nSentiment:'''

response = get_completion(few_shot_prompt)
print(response)

Sentiment: Neutral


**Análisis Crítico:**

**¿Por qué funciona?**
El Few-Shot Prompting proporciona ejemplos concretos que el modelo puede imitar, guiando su comportamiento y reduciendo la ambigüedad en tareas de clasificación o generación.

**Ventaja competitiva:**
Permite adaptar el modelo a tareas específicas sin necesidad de reentrenar, logrando resultados más consistentes y alineados con el criterio deseado.

**Mi contexto:**
Aplicaría esta técnica para clasificar automáticamente opiniones de clientes sobre el servicio, entrenando al modelo con ejemplos reales y mejorando la detección de áreas de mejora en la atención.

### Técnica 3: Chain-of-Thought (Cadena de Pensamiento)

Esta técnica solicita al modelo que explique su razonamiento paso a paso, lo que mejora la precisión en tareas lógicas o de cálculo.

In [50]:
# Example: Chain-of-Thought (Kitchen Stock Calculation)
cot_prompt = '''A restaurant has 120 eggs. Each omelette requires 3 eggs. If 25 omelettes are ordered, how many eggs will remain?\nLet's think step by step.'''

response = get_completion(cot_prompt)
print(response)

Sure! Let's break this problem down step by step:

1. **Start with the total number of eggs:**
   The restaurant has **120 eggs**.

2. **Determine how many eggs are needed for one omelette:**
   Each omelette requires **3 eggs**.

3. **Calculate the total number of eggs needed for 25 omelettes:**
   If 25 omelettes are ordered, the total number of eggs required is:
   \[
   25 \times 3 = 75 \text{ eggs}.
   \]

4. **Subtract the eggs used from the total number of eggs:**
   After making 25 omelettes, the remaining number of eggs is:
   \[
   120 - 75 = 45 \text{ eggs}.
   \]

5. **Final answer:**
   The restaurant will have **45 eggs** remaining.


**Análisis Crítico:**

**¿Por qué funciona?**
La cadena de pensamiento (Chain-of-Thought) obliga al modelo a razonar paso a paso, lo que mejora la precisión en problemas lógicos o de cálculo.

**Ventaja competitiva:**
Evita respuestas precipitadas o incorrectas, ya que el modelo debe justificar cada paso, lo que incrementa la transparencia y la calidad de la solución.

**Mi contexto:**
Utilizaría esta técnica para cálculos de inventario o rentabilidad, asegurando que el modelo explique cómo llega a cada resultado y facilitando la validación de sus recomendaciones.

### Técnica 4: Forzado de Formato / JSON

Se utiliza para obtener respuestas estructuradas en un formato específico (por ejemplo, JSON), facilitando la integración con sistemas automatizados y el procesamiento posterior.

In [51]:
# Example: Format Forcing / JSON (Extracting Dish Data)
json_prompt = '''Extract the following dish information in JSON format with keys: name, ingredients, price.\n\nDish: "Grilled Salmon"\nIngredients: Salmon, lemon, olive oil, salt, pepper\nPrice: $18.50\n\nJSON output:'''

response = get_completion(json_prompt)
print(response)

```json
{
  "name": "Grilled Salmon",
  "ingredients": ["Salmon", "lemon", "olive oil", "salt", "pepper"],
  "price": 18.50
}
```


**Análisis Crítico:**

**¿Por qué funciona?**
El forzado de formato (por ejemplo, JSON) estructura la salida del modelo, facilitando su integración con sistemas automáticos y bases de datos.

**Ventaja competitiva:**
Permite automatizar flujos de trabajo y evita errores de interpretación, ya que la información se presenta de forma clara y estandarizada.

**Mi contexto:**
Usaría esta técnica para extraer y almacenar datos de platos, ventas o tendencias en formato JSON, facilitando el análisis y la visualización en dashboards o sistemas de gestión.

### Técnica 5: Recuperación en Contexto (In-context Retrieval)

Permite al modelo responder preguntas utilizando información relevante proporcionada explícitamente en el prompt, simulando una base de datos temporal.

In [52]:
# Example: In-context Retrieval (Menu of the Day)
menu_text = '''Menu of the Day:\n- Spaghetti Carbonara\n- Grilled Chicken Salad\n- Tomato Soup\n- Chocolate Mousse'''
retrieval_prompt = f"""Based only on the following menu, answer the question.\n\n{menu_text}\n\nQuestion: Is there a vegetarian main course available today?"""

response = get_completion(retrieval_prompt)
print(response)

Yes, the **Tomato Soup** can be considered a vegetarian main course if it is served in a portion large enough to be a main dish and does not contain any non-vegetarian ingredients like chicken stock. However, none of the other listed items are vegetarian main courses.


**Análisis Crítico:**

**¿Por qué funciona?**
La recuperación en contexto permite al modelo responder solo con la información relevante proporcionada, simulando una base de datos temporal y evitando invenciones.

**Ventaja competitiva:**
Aumenta la precisión y la trazabilidad de las respuestas, ya que el modelo se limita a los datos dados y no utiliza información externa o inventada.

**Mi contexto:**
Aplicaría esta técnica para responder preguntas sobre menús, ingredientes o precios, garantizando que las respuestas sean siempre coherentes con la información oficial del restaurante.

### Técnica 6: Dar una "salida" al modelo

Consiste en permitir que el modelo indique cuando no tiene suficiente información para responder, evitando respuestas incorrectas o inventadas.

In [53]:
# Example: Give the model an "out" (Missing Dish Price)
out_prompt = '''If you do not know the answer, say "I don't know based on the information provided."\n\nQuestion: What is the price of the 'Vegan Burger'?\nMenu: Grilled Salmon - $18.50, Caesar Salad - $12.00, Tomato Soup - $7.00'''

response = get_completion(out_prompt)
print(response)

I don't know based on the information provided.


**Análisis Crítico:**

**¿Por qué funciona?**
Dar una "salida" al modelo le permite reconocer sus límites y evitar respuestas incorrectas o inventadas cuando la información es insuficiente.

**Ventaja competitiva:**
Reduce el riesgo de alucinaciones y aumenta la confianza en el sistema, ya que el modelo solo responde cuando tiene datos suficientes.

**Mi contexto:**
Utilizaría esta técnica para consultas sobre platos o precios no registrados, asegurando que el sistema indique claramente cuándo no puede responder en vez de inventar información.

## 1.2 Casos de Uso en el Mundo Real

A continuación se presentan dos implementaciones complejas que combinan múltiples técnicas para resolver problemas reales en el sector restaurantero.

### Técnica 7: Least-to-Most Prompting

**Teoría:**
La técnica "Least-to-Most Prompting" consiste en dividir un problema complejo en subproblemas más pequeños y resolverlos de forma secuencial. Así, el modelo puede abordar tareas difíciles paso a paso, mejorando la calidad del razonamiento y la precisión de la respuesta.

Esta técnica es una forma avanzada de Prompt Splitting, donde dividimos el problema en sub-tareas secuenciales.

Esta técnica fue introducida por investigadores de Google para potenciar el razonamiento complejo en modelos de lenguaje.

In [54]:
# Example: Least-to-Most Prompting (Rentabilidad de un plato)
least_to_most_prompt = (
    "Vamos a calcular la rentabilidad de un nuevo plato en tres pasos.\n\n"
    "Paso 1: Calcula el coste total de los ingredientes si son: pollo (2€), arroz (0.5€), especias (0.2€), verduras (1€).\n"
    "Paso 2: Suma el coste de tiempo de preparación (30 minutos a 10€/hora).\n"
    "Paso 3: Si el precio de mercado es 12€, ¿cuál es el beneficio neto por plato?\n\n"
    "Responde cada paso por separado y da el resultado final."
 )

response = get_completion(
    least_to_most_prompt,
    system_prompt="Eres un experto en gestión de restaurantes. Razona paso a paso."
)
print(response)

¡Claro! Vamos a calcular la rentabilidad del nuevo plato paso a paso.

---

### **Paso 1: Calcular el coste total de los ingredientes**
Los ingredientes y sus costes son:
- Pollo: 2€
- Arroz: 0.5€
- Especias: 0.2€
- Verduras: 1€

**Coste total de los ingredientes:**
\[
2 + 0.5 + 0.2 + 1 = 3.7 \, \text{€}
\]

**Resultado del Paso 1:** El coste total de los ingredientes es **3.7€**.

---

### **Paso 2: Sumar el coste del tiempo de preparación**
El tiempo de preparación es de 30 minutos, y el coste por hora es de 10€/hora. Para calcular el coste de 30 minutos:

\[
\text{Coste de tiempo} = \frac{10}{60} \times 30 = 5 \, \text{€}
\]

**Resultado del Paso 2:** El coste del tiempo de preparación es **5€**.

---

### **Paso 3: Calcular el beneficio neto por plato**
El precio de mercado del plato es 12€. Para calcular el beneficio neto, restamos el coste total (ingredientes + tiempo de preparación) al precio de mercado:

\[
\text{Coste total} = 3.7 + 5 = 8.7 \, \text{€}
\]
\[
\text{Beneficio ne

### Caso A: Service Quality Predictor

Combina el uso de persona (consultor experto) y cadena de pensamiento para predecir la calidad del servicio según la proporción de personal y mesas.

In [55]:
# Case A: Service Quality Predictor (Persona + CoT)
service_system_prompt = "You are a Senior Restaurant Consultant. Predict service quality ratings based on staff-to-table ratio. Explain your reasoning step by step before giving the rating (1-5)."
service_prompt = "A restaurant has 5 staff members and 20 tables. Predict the service quality rating and explain your reasoning."

response = get_completion(service_prompt, system_prompt=service_system_prompt)
print(response)

To predict the service quality rating for a restaurant with 5 staff members and 20 tables, let's break this down step by step:

---

### Step 1: Calculate the staff-to-table ratio
The staff-to-table ratio is calculated as:

\[
\text{Staff-to-Table Ratio} = \frac{\text{Number of Staff}}{\text{Number of Tables}}
\]

\[
\text{Staff-to-Table Ratio} = \frac{5}{20} = 0.25
\]

This means there is 1 staff member for every 4 tables.

---

### Step 2: Assess the implications of the ratio
- **Industry Standards**: In full-service restaurants, a common benchmark is 1 staff member for every 3-4 tables to ensure good service quality. A ratio of 0.25 (1:4) is on the lower end of this range, meaning the staff may be stretched thin, especially during busy periods.
- **Service Impact**: With 1 staff member managing 4 tables, the service quality could be acceptable during off-peak hours but may decline during peak times. Staff may struggle to provide personalized attention, respond quickly to customer ne

**Justificación de Caso A:**

La combinación de la técnica de **Persona** y **Chain-of-Thought** es ideal para consultoría de servicios porque permite simular el razonamiento de un experto real. El modelo no solo responde como un consultor profesional, sino que además explica paso a paso su lógica, lo que aporta transparencia y confianza en la toma de decisiones. Así, se obtienen recomendaciones personalizadas y justificadas, fundamentales para mejorar la calidad del servicio en restaurantes.

### Caso B: Dish Popularity & Trend Analysis

Combina few-shot y forzado de formato JSON para predecir tendencias de popularidad de platillos el próximo mes.

In [56]:
# Case B: Dish Popularity & Trend Analysis (Few-Shot + JSON)
trend_prompt = '''Given the following sales data, predict the top 2 trending dishes for next month.\n\nExample:\nInput: [{"dish": "Pizza Margherita", "sales": 120}, {"dish": "Caesar Salad", "sales": 80}]\nOutput: [{"dish": "Pizza Margherita", "trend": "up"}, {"dish": "Caesar Salad", "trend": "down"}]\n\nInput: [{"dish": "Grilled Salmon", "sales": 90}, {"dish": "Vegan Burger", "sales": 110}, {"dish": "Tomato Soup", "sales": 60}]\nOutput (JSON):'''

response = get_completion(trend_prompt)
print(response)

```json
[
  {"dish": "Vegan Burger", "trend": "up"},
  {"dish": "Grilled Salmon", "trend": "up"}
]
```


**Análisis Crítico:**

**¿Por qué funciona?**
Least-to-Most Prompting permite al modelo abordar problemas complejos de manera ordenada, resolviendo primero los subproblemas más sencillos y construyendo la solución final paso a paso.

**Ventaja competitiva:**
Reduce errores y mejora la precisión en tareas de razonamiento avanzado, ya que el modelo no se pierde en la complejidad global y puede validar cada paso.

**Mi contexto:**
Utilizaría esta técnica para calcular la rentabilidad de nuevos platos o analizar procesos complejos en el restaurante, asegurando que cada variable se tenga en cuenta y el resultado sea fiable.

**Justificación de Caso B:**

La sinergia entre **Few-Shot** y **JSON** es el estándar de oro para exportar tendencias de platos a una base de datos. Few-Shot permite entrenar al modelo con ejemplos reales, asegurando clasificaciones precisas y adaptadas al contexto. El forzado de formato JSON garantiza que los resultados sean estructurados y listos para ser integrados automáticamente en sistemas de análisis o visualización, facilitando la toma de decisiones basada en datos objetivos y actualizados.

---

# **Conclusiones Finales**

**La ingeniería de prompts es la clave para desbloquear el verdadero potencial de los modelos de lenguaje.**

- Permite obtener respuestas más precisas, estructuradas y alineadas con los objetivos del negocio.
- Facilita la integración de la IA en sistemas reales, minimizando errores y alucinaciones.
- Favorece la transparencia y la trazabilidad, esenciales para la toma de decisiones en entornos críticos como la restauración.

